In [ ]:
from pathlib import Path
import json
import math
import os
import sys
from collections import Counter
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "enriched_prs_raw.jsonl").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "enriched_prs_raw.jsonl"
FIGURE_DIR = ROOT / "figures" / "data_understanding"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))

from pipeline.config import load_config
from pipeline.filters import evaluate_example, is_bot_login, is_doc_file, is_generated_or_vendor, is_lockfile, is_meaningful_review_comment, is_source_file, source_file_language

config = load_config(ROOT / "config.yaml")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120

In [ ]:
def parse_datetime(value):
    if not value:
        return pd.NaT
    try:
        return pd.Timestamp(datetime.fromisoformat(str(value).replace("Z", "+00:00")))
    except ValueError:
        return pd.NaT


def month_key(value):
    timestamp = parse_datetime(value)
    if pd.isna(timestamp):
        return None
    return timestamp.strftime("%Y-%m")


def day_key(value):
    timestamp = parse_datetime(value)
    if pd.isna(timestamp):
        return None
    return timestamp.strftime("%Y-%m-%d")


def file_extension(path):
    suffix = Path(path or "").suffix.lower()
    return suffix if suffix else "[none]"


def safe_int(value, default=0):
    if value is None or value == "":
        return default
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def clipped(series, q=0.99):
    upper = series.quantile(q)
    return series.clip(upper=upper), upper


def savefig(name):
    path = FIGURE_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    return path

In [ ]:
records = []
language_counter = Counter()
extension_counter = Counter()
review_state_counter = Counter()
reject_reason_counter = Counter()
api_error_stage_counter = Counter()
json_errors = 0

with DATA_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            json_errors += 1
            continue

        pr = row.get("pr") or {}
        candidate = row.get("candidate") or {}
        files = row.get("files") or []
        reviews = row.get("reviews") or []
        review_comments = row.get("review_comments") or []
        pr_comments = row.get("pr_comments") or []
        linked_issue_comments = row.get("linked_issue_comments") or []
        api_errors = row.get("api_errors") or []
        author = pr.get("user") or {}
        filenames = [item.get("filename") or "" for item in files]
        source_files = [path for path in filenames if is_source_file(path)]
        meaningful_review_comments = [comment for comment in review_comments if is_meaningful_review_comment(comment.get("body"))]
        review_states = [(review.get("state") or "UNKNOWN").upper() for review in reviews]
        has_changes_requested = "CHANGES_REQUESTED" in review_states
        quality = evaluate_example(row, config.dataset, config.filters)

        for path in filenames:
            extension_counter[file_extension(path)] += 1
            if is_source_file(path):
                language_counter[source_file_language(path) or "unknown"] += 1

        for state in review_states:
            review_state_counter[state] += 1

        for reason in quality.get("reject_reasons") or []:
            reject_reason_counter[reason] += 1

        for error in api_errors:
            api_error_stage_counter[error.get("stage") or "unknown"] += 1

        additions = safe_int(pr.get("additions"), safe_int(candidate.get("additions")))
        deletions = safe_int(pr.get("deletions"), safe_int(candidate.get("deletions")))
        changed_files = safe_int(pr.get("changed_files"), safe_int(candidate.get("changed_files")))
        linked_issue_present = bool(row.get("linked_issue_number") and row.get("linked_issue"))

        records.append({
            "repo": row.get("repo_name"),
            "pr_number": row.get("pr_number"),
            "author": author.get("login"),
            "author_type": author.get("type") or "UNKNOWN",
            "author_association": pr.get("author_association") or "UNKNOWN",
            "is_bot_author": is_bot_login(author.get("login")),
            "created_month": month_key(pr.get("created_at")),
            "merged_month": month_key(pr.get("merged_at") or pr.get("closed_at")),
            "retrieved_day": day_key(row.get("retrieved_at")),
            "has_pr": bool(pr),
            "has_files": bool(files),
            "has_reviews": bool(reviews),
            "has_review_comments": bool(review_comments),
            "has_pr_comments": bool(pr_comments),
            "has_full_diff": bool(row.get("full_diff")),
            "has_linked_issue": linked_issue_present,
            "has_source_file": bool(source_files),
            "api_error_count": len(api_errors),
            "changed_files": changed_files,
            "additions": additions,
            "deletions": deletions,
            "diff_lines": additions + deletions,
            "commits": safe_int(pr.get("commits")),
            "file_count": len(files),
            "source_file_count": len(source_files),
            "review_count": len(reviews),
            "review_comment_count": len(review_comments),
            "meaningful_review_comment_count": len(meaningful_review_comments),
            "pr_comment_count": len(pr_comments),
            "linked_issue_comment_count": len(linked_issue_comments),
            "full_diff_chars": len(row.get("full_diff") or ""),
            "review_concern": has_changes_requested or len(meaningful_review_comments) >= config.dataset.min_meaningful_review_comments,
            "accepted_current_filters": bool(quality.get("accepted")),
            "quality_score": quality.get("score", 0),
        })

df = pd.DataFrame.from_records(records)
file_size_gb = DATA_PATH.stat().st_size / 1024 ** 3

In [ ]:
summary = pd.Series({
    "records": len(df),
    "json_errors": json_errors,
    "file_size_gb": round(file_size_gb, 2),
    "repositories": df["repo"].nunique(),
    "authors": df["author"].nunique(),
    "linked_issue_records": int(df["has_linked_issue"].sum()),
    "linked_issue_rate": round(df["has_linked_issue"].mean(), 4),
    "accepted_current_filters": int(df["accepted_current_filters"].sum()),
    "acceptance_rate": round(df["accepted_current_filters"].mean(), 4),
    "review_concern_positive_rate": round(df["review_concern"].mean(), 4),
    "api_error_rows": int((df["api_error_count"] > 0).sum()),
})
summary

In [ ]:
completeness_table = pd.Series({
    "Pull request object": df["has_pr"].mean(),
    "File list": df["has_files"].mean(),
    "Full diff": df["has_full_diff"].mean(),
    "Formal reviews": df["has_reviews"].mean(),
    "Review comments": df["has_review_comments"].mean(),
    "PR comments": df["has_pr_comments"].mean(),
    "Source files": df["has_source_file"].mean(),
    "Explicit linked issue": df["has_linked_issue"].mean(),
}).rename("rate").to_frame()
completeness_table["percent"] = completeness_table["rate"] * 100
completeness_table

In [ ]:
numeric_summary = df[[
    "changed_files",
    "diff_lines",
    "additions",
    "deletions",
    "commits",
    "review_count",
    "review_comment_count",
    "meaningful_review_comment_count",
    "pr_comment_count",
    "source_file_count",
    "full_diff_chars",
]].describe(percentiles=[0.5, 0.9, 0.99]).T
numeric_summary

In [ ]:
top_repositories = df["repo"].value_counts().head(20).rename_axis("repo").reset_index(name="records")
top_languages = pd.DataFrame(language_counter.most_common(20), columns=["language", "files"])
top_extensions = pd.DataFrame(extension_counter.most_common(20), columns=["extension", "files"])
top_reject_reasons = pd.DataFrame(reject_reason_counter.most_common(20), columns=["reason", "records"])
review_states = pd.DataFrame(review_state_counter.most_common(), columns=["state", "reviews"])
top_repositories, top_languages, top_extensions, top_reject_reasons, review_states

In [ ]:
timeline = pd.concat([
    df["created_month"].value_counts().rename("created"),
    df["merged_month"].value_counts().rename("merged"),
], axis=1).fillna(0).sort_index()
fig, ax = plt.subplots(figsize=(11, 4))
timeline.plot(kind="bar", ax=ax, width=0.85)
ax.set_xlabel("Month")
ax.set_ylabel("Pull requests")
ax.set_title("Collection timeline")
ax.tick_params(axis="x", rotation=60)
savefig("collection_timeline.png")

In [ ]:
plot_data = completeness_table.reset_index().rename(columns={"index": "field"}).sort_values("percent")
fig, ax = plt.subplots(figsize=(8, 4.8))
sns.barplot(data=plot_data, x="percent", y="field", ax=ax, color="#4C78A8")
ax.set_xlim(0, 100)
ax.set_xlabel("Completeness, percent")
ax.set_ylabel("")
ax.set_title("Schema completeness")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=3)
savefig("schema_completeness.png")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
size_columns = [
    ("changed_files", "Changed files"),
    ("diff_lines", "Changed lines"),
    ("additions", "Additions"),
    ("source_file_count", "Source files"),
]
for ax, (column, label) in zip(axes.ravel(), size_columns):
    values, upper = clipped(df[column], 0.99)
    sns.histplot(values, bins=35, ax=ax, color="#4C78A8")
    ax.set_title(label)
    ax.set_xlabel(f"Value, clipped at p99={upper:.0f}")
    ax.set_ylabel("Records")
savefig("size_distributions.png")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
activity_columns = [
    ("review_count", "Formal reviews"),
    ("review_comment_count", "Review comments"),
    ("meaningful_review_comment_count", "Meaningful review comments"),
    ("pr_comment_count", "Pull request comments"),
]
for ax, (column, label) in zip(axes.ravel(), activity_columns):
    values, upper = clipped(df[column], 0.99)
    sns.histplot(values, bins=35, ax=ax, color="#59A14F")
    ax.set_title(label)
    ax.set_xlabel(f"Count, clipped at p99={upper:.0f}")
    ax.set_ylabel("Records")
savefig("review_activity.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=top_languages.head(12), x="files", y="language", ax=axes[0], color="#F28E2B")
sns.barplot(data=top_extensions.head(12), x="files", y="extension", ax=axes[1], color="#E15759")
axes[0].set_title("Top source languages")
axes[0].set_xlabel("Files")
axes[0].set_ylabel("")
axes[1].set_title("Top file extensions")
axes[1].set_xlabel("Files")
axes[1].set_ylabel("")
savefig("language_distribution.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
acceptance = pd.DataFrame({
    "status": ["Accepted", "Rejected"],
    "records": [int(df["accepted_current_filters"].sum()), int((~df["accepted_current_filters"]).sum())],
})
sns.barplot(data=acceptance, x="status", y="records", ax=axes[0], palette=["#59A14F", "#E15759"], hue="status", legend=False)
sns.barplot(data=top_reject_reasons.head(10), x="records", y="reason", ax=axes[1], color="#E15759")
axes[0].set_title("Current filter outcome")
axes[0].set_xlabel("")
axes[0].set_ylabel("Records")
axes[1].set_title("Top rejection reasons")
axes[1].set_xlabel("Records")
axes[1].set_ylabel("")
savefig("quality_filters.png")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=top_repositories.head(15), x="records", y="repo", ax=ax, color="#4C78A8")
ax.set_title("Repository concentration")
ax.set_xlabel("Records")
ax.set_ylabel("")
savefig("repo_concentration.png")

In [ ]:
expected_plots = [
    "collection_timeline.png",
    "schema_completeness.png",
    "size_distributions.png",
    "review_activity.png",
    "language_distribution.png",
    "quality_filters.png",
    "repo_concentration.png",
]
plot_sizes = {name: (FIGURE_DIR / name).stat().st_size for name in expected_plots}
plot_sizes